# SQUAREQ: Earthquake Alert Multi-Classification

This notebook compares two models for earthquake alert type prediction (green, yellow, orange, red):

1. **SQUAREQ**: SAC-optimized Quantum Support Vector Classifier
2. **Classical SVC**: RBF kernel Support Vector Classifier

## Dataset
- **Source**: `earthquake_alert_balanced_dataset.csv`
- **Target**: Multi-class classification (alert types: green, yellow, orange, red)
- **Features**: 6 features selected via multi-method feature selection (MI + F-test + RF)
- **Splits**: Train (70%) / Validation (15%) / Test (15%)


In [ ]:
import sys
import os
sys.path.append('../src')

import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.svm import SVC

from squareq_core import (
    FixedQMetricsQuantumEnvironment,
    SACAgent,
    create_quantum_circuit_from_params
)
from utils import load_earthquake_alert_data, train_multiclass_quantum_qsvc

print("✅ Imports successful")


## 1. Load and Prepare Data


In [ ]:
# Load earthquake alert data
data_path = '../data/earthquake_alert_balanced_dataset.csv'
X, y, selected_features, label_encoder = load_earthquake_alert_data(data_path)

print(f"Dataset shape: {X.shape}")
print(f"Selected features: {selected_features}")
print(f"Class distribution: {np.bincount(y)}")
print(f"\nLabel mapping:")
for label, idx in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"  {idx} -> {label}")

# Split data: Train (70%) / Val (15%) / Test (15%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, train_size=0.7, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"\nSplits:")
print(f"  Train: {len(y_train)} samples")
print(f"  Val:   {len(y_val)} samples")
print(f"  Test:  {len(y_test)} samples")


## 2. Train SAC Agent to Optimize Quantum Circuit


In [ ]:
# Use subset for SAC training (500 samples)
sac_samples = min(500, len(y_train))
idx = np.random.choice(len(y_train), sac_samples, replace=False)
X_sac = X_train[idx]
y_sac = y_train[idx]

print(f"Training SAC on {sac_samples} samples...")
print(f"Episodes: 100, Steps per episode: 5")

# Create environment and agent
env = FixedQMetricsQuantumEnvironment(X_sac, y_sac, num_qubits=X_sac.shape[1])
agent = SACAgent(env.state_dim, env.action_dim)

# Train SAC
episodes = 100
rewards = []
best_reward = -float('inf')
best_params = None

for episode in range(episodes):
    state = env.reset()
    total_reward = 0
    
    for step in range(5):
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        
        agent.replay_buffer.push(state, action, reward, next_state, done)
        agent.update()
        
        state = next_state
        total_reward += reward
        
        if done:
            break
    
    rewards.append(total_reward)
    
    if total_reward > best_reward:
        best_reward = total_reward
        best_params = {
            'threshold': (action[0] + 1) / 2,
            'theta_values': action[1:env.num_qubits+1],
            'gate_types': action[env.num_qubits+1:env.num_qubits*2+1],
            'entanglement_strength': action[env.num_qubits*2+1:],
            'num_qubits': env.num_qubits
        }
    
    if (episode + 1) % 20 == 0:
        print(f"  Episode {episode + 1:3d}: Reward = {total_reward:.4f}, Best = {best_reward:.4f}")

print(f"\n✅ SAC training complete!")
print(f"   Best reward: {best_reward:.4f}")
print(f"   Average reward: {np.mean(rewards):.4f}")


## 3. Model 1: SQUAREQ (SAC-Optimized QSVC)


In [ ]:
print("Training SQUAREQ QSVC (multi-class)...")
qsvc_metrics = train_multiclass_quantum_qsvc(
    X_train, y_train, X_val, y_val, X_test, y_test, best_params
)

print(f"✅ SQUAREQ Results:")
print(f"   Train Accuracy: {qsvc_metrics['train_accuracy']:.4f}")
print(f"   Val Accuracy:   {qsvc_metrics['val_accuracy']:.4f}")
print(f"   Test Accuracy:  {qsvc_metrics['test_accuracy']:.4f}")
print(f"   Test Precision: {qsvc_metrics['test_precision']:.4f}")
print(f"   Test Recall:    {qsvc_metrics['test_recall']:.4f}")
print(f"   Test F1:        {qsvc_metrics['test_f1']:.4f}")
print(f"   Training Time:  {qsvc_metrics['train_time']:.2f}s")


## 4. Model 2: Classical SVC (RBF Kernel)


In [ ]:
# Create classical SVC
svc_classical = SVC(kernel='rbf', random_state=42)

# Train
print("Training Classical SVC (RBF)...")
start_time = time.time()
svc_classical.fit(X_train, y_train)
train_time_svc = time.time() - start_time

# Evaluate
y_train_pred_svc = svc_classical.predict(X_train)
y_val_pred_svc = svc_classical.predict(X_val)
y_test_pred_svc = svc_classical.predict(X_test)

train_acc_svc = accuracy_score(y_train, y_train_pred_svc)
val_acc_svc = accuracy_score(y_val, y_val_pred_svc)
test_acc_svc = accuracy_score(y_test, y_test_pred_svc)

prec_svc, rec_svc, f1_svc, _ = precision_recall_fscore_support(
    y_test, y_test_pred_svc, average='macro', zero_division=0
)

print(f"✅ Classical SVC Results:")
print(f"   Train Accuracy: {train_acc_svc:.4f}")
print(f"   Val Accuracy:   {val_acc_svc:.4f}")
print(f"   Test Accuracy:  {test_acc_svc:.4f}")
print(f"   Test Precision: {prec_svc:.4f}")
print(f"   Test Recall:    {rec_svc:.4f}")
print(f"   Test F1:        {f1_svc:.4f}")
print(f"   Training Time:  {train_time_svc:.2f}s")


## 5. Comparison and Results Summary


In [ ]:
# Create comparison DataFrame
results = pd.DataFrame({
    'Model': ['SQUAREQ (SAC-optimized)', 'Classical SVC (RBF)'],
    'Train Accuracy': [qsvc_metrics['train_accuracy'], train_acc_svc],
    'Val Accuracy': [qsvc_metrics['val_accuracy'], val_acc_svc],
    'Test Accuracy': [qsvc_metrics['test_accuracy'], test_acc_svc],
    'Test Precision': [qsvc_metrics['test_precision'], prec_svc],
    'Test Recall': [qsvc_metrics['test_recall'], rec_svc],
    'Test F1': [qsvc_metrics['test_f1'], f1_svc],
    'Training Time (s)': [qsvc_metrics['train_time'], train_time_svc]
})

print("=" * 80)
print("COMPARISON RESULTS")
print("=" * 80)
print(results.to_string(index=False))
print("=" * 80)

# Detailed metrics
print("\n\nDETAILED METRICS (Test Set):")
print("\n" + "-" * 80)
print("SQUAREQ (SAC-optimized):")
print(qsvc_metrics['classification_report'])

print("\n" + "-" * 80)
print("Classical SVC (RBF):")
print(classification_report(y_test, y_test_pred_svc, 
                          target_names=label_encoder.classes_, zero_division=0))


In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
models = ['SQUAREQ', 'Classical SVC']
train_accs = [qsvc_metrics['train_accuracy'], train_acc_svc]
val_accs = [qsvc_metrics['val_accuracy'], val_acc_svc]
test_accs = [qsvc_metrics['test_accuracy'], test_acc_svc]

x = np.arange(len(models))
width = 0.25

axes[0].bar(x - width, train_accs, width, label='Train', alpha=0.8)
axes[0].bar(x, val_accs, width, label='Val', alpha=0.8)
axes[0].bar(x + width, test_accs, width, label='Test', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Runtime comparison
times = [qsvc_metrics['train_time'], train_time_svc]
axes[1].bar(models, times, alpha=0.8, color=['#1f77b4', '#2ca02c'])
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Training Time (s)')
axes[1].set_title('Training Time Comparison')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
